# SENTINEL — chạy thử bản gốc trên Google Colab

Notebook này chạy bản gốc của repo
[anshulec23-cloud/digital-surveillance-system](https://github.com/anshulec23-cloud/digital-surveillance-system)
(YOLOv8 → DeepSORT → LSTM Autoencoder) với video tổng hợp của tác giả.

**Trước khi chạy:** chọn `Runtime → Change runtime type → GPU` (nếu muốn chạy chế độ GPU).

**Lưu ý bảo mật**
- Chỉ clone nhánh `main` và ghim commit `66d81da` (đã được đọc và kiểm tra).
  Các nhánh `ci-sync-*` trên repo gốc có workflow đánh cắp secrets, **không dùng**.
- README hướng dẫn tải `lstm_autoencoder.pt` từ Release v1.0.0, nhưng file đó **không tồn tại**.
  Release chỉ có một file `.exe` lạ, **không tải và không chạy file này**. Trọng số LSTM được train lại ở bước 5.

**Lỗi đã biết của bản gốc**
1. `LSTMAutoencoder.load()` không chuyển model sang `device`, nên `YOLO_DEVICE=cuda` sẽ crash
   (*Expected all tensors to be on the same device*). Với `USE_GPU=True`, notebook vá tạm 1 dòng trên bản clone ở Colab.
2. `python scripts/evaluate.py` lỗi import `core`, nên notebook chạy kèm `PYTHONPATH=.`.

## 0. Cấu hình

In [ ]:
REPO_URL = "https://github.com/anshulec23-cloud/digital-surveillance-system.git"
BRANCH   = "main"
COMMIT   = "66d81daa6daaae9129d1c0e9c9fb3b62bc4dc686"  # commit đã kiểm tra
WORKDIR  = "/content/digital-surveillance-system"

# False = đúng code và cấu hình gốc của tác giả (CPU).
# True  = chạy GPU, vá 1 dòng .to(device) trong anomaly/model.py (chỉ trên bản clone này).
USE_GPU = True

VIDEO_DURATION = 30        # giây, video tổng hợp
TRAIN_EPOCHS   = 30
TRAIN_SAMPLES  = 2000
# README của tác giả: sau khi train thì cập nhật ANOMALY_THRESHOLD theo giá trị gợi ý
USE_SUGGESTED_THRESHOLD = True

## 1. Kiểm tra GPU

In [ ]:
!nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv || echo "Không có GPU"
import torch
print("torch", torch.__version__, "| CUDA available:", torch.cuda.is_available())
if USE_GPU and not torch.cuda.is_available():
    print("⚠️ USE_GPU=True nhưng runtime không có GPU, chuyển sang CPU.")
    USE_GPU = False

## 2. Clone bản gốc (chỉ nhánh `main`, ghim commit)

In [ ]:
import os, subprocess
if not os.path.isdir(WORKDIR):
    !git clone --branch {BRANCH} --single-branch --no-tags {REPO_URL} {WORKDIR}
%cd {WORKDIR}
!git checkout -q {COMMIT}
!git log --oneline -1

## 3. Cài thư viện

Colab đã có sẵn `torch` bản CUDA, và `requirements.txt` chỉ yêu cầu `torch>=2.1` nên pip sẽ giữ nguyên bản có sẵn.

In [ ]:
!pip install -q -r requirements.txt pytest
!python -c "import ultralytics, deep_sort_realtime, fastapi, torch, cv2; print('ultralytics', ultralytics.__version__, '| torch', torch.__version__, '| cv2', cv2.__version__)"

## 4. Cấu hình `.env` (và vá lỗi device nếu chạy GPU)

In [ ]:
from pathlib import Path

if USE_GPU:
    p = Path("anomaly/model.py")
    src = p.read_text()
    old, new = "model = cls(**cfg)", "model = cls(**cfg).to(device)"
    if new not in src:
        assert src.count(old) == 1, "Không tìm thấy chỗ cần vá, code gốc đã thay đổi?"
        p.write_text(src.replace(old, new))
    print("Đã vá anomaly/model.py:")
    !git diff --stat

env = {"YOLO_DEVICE": "cuda" if USE_GPU else "cpu"}
def write_env():
    Path(".env").write_text("".join(f"{k}={v}\n" for k, v in env.items()))
    print(Path(".env").read_text())
write_env()

## 5. Chạy test của tác giả

In [ ]:
!python -m pytest tests -q

## 6. Tạo video tổng hợp (`scripts/generate_test_video.py`)

In [ ]:
!python scripts/generate_test_video.py --out test_video.mp4 --duration {VIDEO_DURATION}

## 7. Train LSTM Autoencoder bằng dữ liệu tổng hợp

Thay cho trọng số pre-trained không tồn tại. Output: `models/lstm_autoencoder.pt`.

In [ ]:
import re
out = subprocess.run(
    ["python", "main.py", "train", "--synthetic",
     "--epochs", str(TRAIN_EPOCHS), "--samples", str(TRAIN_SAMPLES)],
    capture_output=True, text=True,
)
print(out.stdout[-3000:], out.stderr[-3000:])
out.check_returncode()

m = re.search(r"Suggested anomaly_threshold = ([0-9.]+)", out.stdout)
if m and USE_SUGGESTED_THRESHOLD:
    env["ANOMALY_THRESHOLD"] = m.group(1)
    write_env()

## 8. Chạy pipeline headless trên video tổng hợp

`cv2.imshow` không chạy được trên Colab nên dùng `--no-display`, rồi lưu video kết quả.

> Video tổng hợp chỉ gồm các hình elip, không phải người thật, nên YOLO (COCO "person") có thể phát hiện được ít hoặc không phát hiện được gì.
> Nếu `Total alerts` = 0 và không có box nào thì đó là giới hạn của dữ liệu tổng hợp, chưa chắc là lỗi.

In [ ]:
!rm -f logs/events.jsonl output.mp4
!python main.py run --source test_video.mp4 --save output.mp4 --no-display 2>&1 | tail -n 25

### Xem video kết quả

In [ ]:
# mp4v không phát được trên trình duyệt, nên chuyển sang H.264 để xem inline
!ffmpeg -y -loglevel error -i output.mp4 -vcodec libx264 -crf 28 -pix_fmt yuv420p output_h264.mp4
from IPython.display import Video
Video("output_h264.mp4", embed=True, width=800)

### Thống kê sự kiện bất thường (`logs/events.jsonl`)

In [ ]:
import pandas as pd
log = Path("logs/events.jsonl")
if log.exists() and log.stat().st_size:
    df = pd.read_json(log, lines=True)
    display(df["anomaly_type"].value_counts().rename("count").to_frame())
    display(df.head(20))
else:
    print("Không có sự kiện nào.")

## 9. Đánh giá / đo FPS (`scripts/evaluate.py`)

Ở chế độ `--synthetic`, script dùng **chính dự đoán làm ground truth**, nên Precision/Recall/F1 luôn ≈ 1.
Chỉ có **FPS** là con số có ý nghĩa.

In [ ]:
!PYTHONPATH=. python scripts/evaluate.py --video test_video.mp4 --synthetic 2>&1 | tail -n 20

## 10. Dashboard web (FastAPI) qua proxy của Colab

Chạy `python main.py serve` ở nền, rồi mở cổng 8000 qua proxy của Colab.
Trên dashboard, upload `test_video.mp4` (tải về từ panel Files bên trái) hoặc dùng cell bên dưới để upload qua API.

In [ ]:
import time, urllib.request
server = subprocess.Popen(
    ["python", "main.py", "serve", "--port", "8000"],
    stdout=open("server.log", "w"), stderr=subprocess.STDOUT,
)
for _ in range(60):
    try:
        urllib.request.urlopen("http://127.0.0.1:8000/api/status", timeout=1)
        print("Server đã sẵn sàng")
        break
    except Exception:
        time.sleep(1)
else:
    print(open("server.log").read()[-3000:])

from google.colab import output
output.serve_kernel_port_as_window(8000, path="/")

Upload video qua API (sau khi đã mở dashboard để xem stream):

In [ ]:
!curl -s -X POST -F "file=@test_video.mp4;type=video/mp4" http://127.0.0.1:8000/api/upload; echo
!sleep 5; curl -s http://127.0.0.1:8000/api/status; echo

Dừng server:

In [ ]:
server.terminate(); server.wait(timeout=10)
!tail -n 20 server.log

## 11. (Tuỳ chọn) Tải kết quả về máy

In [ ]:
!zip -q -j sentinel_results.zip output_h264.mp4 logs/events.jsonl models/lstm_autoencoder.pt .env
from google.colab import files
files.download("sentinel_results.zip")